# Agent benchmark: causal effects of a job-training program

Five libraries were each given to a Claude Haiku agent with the same brief: a
non-randomized job-training program, follow-up earnings missing for 16% of
applicants, and seven questions an agency would ask, from the average effect
to whom to enrol next year. The outcome is simulated from a known structural
model on real survey covariates, so every effect, interval and claim is graded
against the truth on ten hidden replicates, including two where the program
does nothing. The scores use the knowledge-discovery scorecard
(`docs/scorecard/scorecard.md`) with thresholds fixed before any analyst
started.

The mboostrs column is the fourth mboostrs analyst, run after three rounds of
library fixes prompted by the earlier runs. The other four libraries ran once.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import polars as pl
from great_tables import GT, loc, style
from IPython.display import Markdown, display

ROOT = Path("causal_discovery")
sys.path.insert(0, str(ROOT))
import grade  # noqa: E402  (the pre-registered grader, docs/book/benchmarks/causal_discovery/grade.py)

VARIANT = "after"
LIBS = {
    "mboostrs": "mboostrs 0.1.3",
    "econml": "EconML 0.17.0",
    "doubleml": "DoubleML 0.11.4",
    "statsmodels": "statsmodels 0.15.0",
    "lightgbm": "LightGBM 4.7.0",
}
# The mboostrs column is the first run ("before") or the rerun after the fixes
# ("after"); the other libraries ran once and are the same on both pages.
EVAL = {lib: ROOT / "eval" / lib for lib in LIBS}
RUNS = {lib: ROOT / "runs" / lib for lib in LIBS}
if VARIANT == "after":
    EVAL["mboostrs"] = ROOT / "rerun3" / "eval" / "mboostrs"
    RUNS["mboostrs"] = ROOT / "rerun3" / "mboostrs"
TRUTH = ROOT / "truth"
KEY = json.loads((TRUTH / "key.json").read_text())
HIDDEN = ["H1", "H2", "H3", "H4", "H5", "H6", "N1", "N2", "S1", "S2"]
ALT = [r for r in HIDDEN if not r.startswith("N")]

per = {lib: {rep: grade.grade_rep(EVAL[lib] / rep, TRUTH / "reps" / rep, KEY) for rep in HIDDEN}
       for lib in LIBS}
summary = {lib: grade.summarise(per[lib]) for lib in LIBS}
runlogs = {lib: json.loads((EVAL[lib] / "runlog.json").read_text()) for lib in LIBS}
JUDGED = json.loads((ROOT / f"judged_scores_after.json").read_text())
dims = ["D1", "D2", "D3", "D4", "D5", "D6", "D7", "D8", "D9", "D10"]
names = {"D1": "Relationship discovery", "D2": "Causal humility", "D3": "Explanation quality",
         "D4": "Importance quantification", "D5": "Iteration speed", "D6": "Robustness",
         "D7": "Human-friendliness", "D8": "Agent-friendliness", "D9": "Uncertainty honesty",
         "D10": "Reproducibility"}

## Headline results

The table compares the five pipelines on the hidden replicates. Bold marks the
best value in each column (closest to 0.90 for coverage columns).

In [2]:
rows = []
for lib in LIBS:
    s = summary[lib]
    rows.append({
        "library": LIBS[lib],
        "ate_cov": s.get("coverage", {}).get("ATE"),
        "ate_rmse": s.get("rmse", {}).get("ATE"),
        "q_cov": s.get("coverage_all_queries"),
        "null_false": s.get("null_false_claims"),
        "pehe": s.get("pehe"),
        "cate_cov": s.get("cate_coverage"),
        "regret": s.get("policy_regret_share"),
        "roles": s.get("role_accuracy"),
        "mod_tpr": s.get("modifier_tpr"),
        "channels": s.get("channels_correct"),
        "overlap": s.get("overlap_named"),
        "failed": s["replicates"] - s["ran"],
    })
headline = pl.DataFrame(rows)


def bold_best(gt, frame, col, rule):
    values = frame[col].to_numpy().astype(float)
    if not np.isfinite(values).any():
        return gt
    if rule == "max":
        target = np.nanmax(values)
    elif rule == "nominal":
        target = values[np.nanargmin(np.abs(values - 0.90))]
    else:
        target = np.nanmin(values)
    rows_ = [i for i, v in enumerate(values) if np.isfinite(v) and abs(v - target) < 1e-9]
    return gt.tab_style(style.text(weight="bold"), loc.body(columns=[col], rows=rows_))


gt = (
    GT(headline, rowname_col="library")
    .tab_header(
        title="Five libraries, each run by an agent analyst, on a job-training evaluation with a known answer",
        subtitle="Hidden replicates H1-H6 and S1-S2 (effects), N1-N2 (no effect). Coverage and shares in [0, 1]; "
                 "RMSE and PEHE in dollars of follow-up earnings; regret 0 = best ranking, 1 = random.")
    .tab_spanner("Average effect", ["ate_cov", "ate_rmse"])
    .tab_spanner("All 8 effect queries", ["q_cov", "null_false"])
    .tab_spanner("Per-applicant effects", ["pehe", "cate_cov", "regret"])
    .tab_spanner("Causal structure", ["roles", "mod_tpr", "channels", "overlap"])
    .cols_label(ate_cov="90% coverage", ate_rmse="RMSE", q_cov="90% coverage",
                null_false="False claims, no-effect data", pehe="PEHE", cate_cov="90% coverage",
                regret="Top-20% regret", roles="Roles correct", mod_tpr="Modifiers found",
                channels="Channels correct", overlap="Overlap named", failed="Failed runs")
    .fmt_number(["ate_cov", "q_cov", "cate_cov", "regret", "roles", "mod_tpr", "channels", "overlap"], decimals=2)
    .fmt_number(["null_false"], decimals=1)
    .fmt_integer(["ate_rmse", "pehe"])
    .sub_missing(missing_text="none")
    .tab_source_note("ATE and query coverage: share of replicates whose 90% interval contains the true "
                     "average effect (8 replicates per query). A missing bound counts as not covering.")
    .tab_source_note("False claims, no-effect data: mean over N1-N2 of queries whose interval excludes zero, "
                     "effect modifiers claimed and channels claimed.")
    .tab_source_note("PEHE: root mean squared error of the per-applicant effect against the true expected "
                     "effect on 4,000 new applicants. Top-20% regret: (oracle top-20% mean effect - chosen "
                     "top-20% mean effect) / (oracle - random).")
    .tab_source_note("Roles correct: share of the 16 columns given the key's role. Modifiers found: share of the "
                     "4 material effect modifiers claimed. Overlap named: limited_overlap names prior_earnings.")
)
for col, rule in [("ate_cov", "nominal"), ("ate_rmse", "min"), ("q_cov", "nominal"), ("null_false", "min"),
                  ("pehe", "min"), ("cate_cov", "nominal"), ("regret", "min"), ("roles", "max"),
                  ("mod_tpr", "max"), ("channels", "max"), ("overlap", "max")]:
    gt = bold_best(gt, headline, col, rule)
gt

GT(_tbl_data=shape: (5, 13)
┌───────────────────┬─────────┬─────────────┬──────────┬───┬─────────┬──────────┬─────────┬────────┐
│ library           ┆ ate_cov ┆ ate_rmse    ┆ q_cov    ┆ … ┆ mod_tpr ┆ channels ┆ overlap ┆ failed │
│ ---               ┆ ---     ┆ ---         ┆ ---      ┆   ┆ ---     ┆ ---      ┆ ---     ┆ ---    │
│ str               ┆ f64     ┆ f64         ┆ f64      ┆   ┆ f64     ┆ f64      ┆ f64     ┆ i64    │
╞═══════════════════╪═════════╪═════════════╪══════════╪═══╪═════════╪══════════╪═════════╪════════╡
│ mboostrs 0.1.3    ┆ 0.875   ┆ 3250.007233 ┆ 0.90625  ┆ … ┆ 0.0625  ┆ 0.8125   ┆ 1.0     ┆ 0      │
│ EconML 0.17.0     ┆ 0.875   ┆ 1157.382107 ┆ 0.765625 ┆ … ┆ 0.0     ┆ 0.8125   ┆ 1.0     ┆ 0      │
│ DoubleML 0.11.4   ┆ 0.75    ┆ 2033.105164 ┆ 0.875    ┆ … ┆ 0.03125 ┆ 0.625    ┆ 0.875   ┆ 0      │
│ statsmodels       ┆ 0.875   ┆ 593.332176  ┆ 0.796875 ┆ … ┆ 0.0     ┆ 0.4375   ┆ 1.0     ┆ 0      │
│ 0.15.0            ┆         ┆             ┆          ┆   ┆         ┆          ┆         ┆        │
│ LightGBM 4.7.0    ┆ 0.5     ┆ 2707.973718 ┆ 0.734375 ┆ … ┆ 0.21875 ┆ 0.6875   ┆ 1.0     ┆ 0      │
└───────────────────┴─────────┴─────────────┴──────────┴───┴─────────┴──────────┴─────────┴────────┘, _body=<great_tables._gt_data.Body object at 0x10bac1610>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='ate_cov', type=<ColInfoTypeEnum.default: 1>, column_label='90% coverage', column_align='right', column_width=None), ColInfo(var='ate_rmse', type=<ColInfoTypeEnum.default: 1>, column_label='RMSE', column_align='right', column_width=None), ColInfo(var='q_cov', type=<ColInfoTypeEnum.default: 1>, column_label='90% coverage', column_align='right', column_width=None), ColInfo(var='null_false', type=<ColInfoTypeEnum.default: 1>, column_label='False claims, no-effect data', column_align='right', column_width=None), ColInfo(var='pehe', type=<ColInfoTypeEnum.default: 1>, column_label='PEHE', column_align='right', column_width=None), ColInfo(var='cate_cov', type=<ColInfoTypeEnum.default: 1>, column_label='90% coverage', column_align='right', column_width=None), ColInfo(var='regret', type=<ColInfoTypeEnum.default: 1>, column_label='Top-20% regret', column_align='right', column_width=None), ColInfo(var='roles', type=<ColInfoTypeEnum.default: 1>, column_label='Roles correct', column_align='right', column_width=None), ColInfo(var='mod_tpr', type=<ColInfoTypeEnum.default: 1>, column_label='Modifiers found', column_align='right', column_width=None), ColInfo(var='channels', type=<ColInfoTypeEnum.default: 1>, column_label='Channels correct', column_align='right', column_width=None), ColInfo(var='overlap', type=<ColInfoTypeEnum.default: 1>, column_label='Overlap named', column_align='right', column_width=None), ColInfo(var='failed', type=<ColInfoTypeEnum.default: 1>, column_label='Failed runs', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10ae3efc0>, _spanners=Spanners([SpannerInfo(spanner_id='Average effect', spanner_level=0, spanner_label='Average effect', spanner_units=None, spanner_pattern=None, vars=['ate_cov', 'ate_rmse'], built=None), SpannerInfo(spanner_id='All 8 effect queries', spanner_level=0, spanner_label='All 8 effect queries', spanner_units=None, spanner_pattern=None, vars=['q_cov', 'null_false'], built=None), SpannerInfo(spanner_id='Per-applicant effects', spanner_level=0, spanner_label='Per-applicant effects', spanner_units=None, spanner_pattern=None, vars=['pehe', 'cate_cov', 'regret'], built=None), SpannerInfo(spanner_id='Causal structure', spanner_level=0, spanner_label='Causal structure', spanner_units=None, spanner_pattern=None, vars=['roles', 'mod_tpr', 'channels', 'overlap'], built=None)]), _heading=Heading(title='Five libraries, each run by an agent analyst, on a job-training evaluation with a known answer', subtitle='Hidden replicates H1-H6 and S1-S2 (eff

No library dominates. statsmodels, with a hand-built R-learner and inverse
response weights, has the most accurate average effect (RMSE 593 dollars) and
per-applicant effects (PEHE 1,985). EconML's `LinearDML` has the best
structure scores (roles 0.94, a valid adjustment set on every replicate) and
the best ranking (regret 0.36). mboostrs has the best coverage of the eight
effect queries (0.91), no false claims on the no-effect data, and a valid
adjustment set on five of eight replicates, but its ATE error (3,250 dollars)
is the largest of the five and its per-applicant intervals cover 0.66 of the
true effects. LightGBM, with meta-learners written by hand, has the largest
per-applicant error and the most false claims.

## Experiment setting

The task is a program evaluation of the kind a workforce agency commissions.
A voluntary job-training program enrolled some applicants and not others;
enrolment was steered towards low earners and rare above 40,000 dollars of
prior earnings; a follow-up survey a year later recorded earnings, and 16% of
applicants did not answer it. The agency asks for the average effect, the
channel it worked through, who gained more, which columns to adjust for,
where the data cannot answer, a ranking of next year's applicants, and the
sensitivity to an unmeasured confounder.

The applicant characteristics are the 28,155 men in the March 1988 Current
Population Survey (`AER::CPS1988`): education, experience, ethnicity,
metropolitan residence, region, part-time work, and prior earnings derived
from the real weekly wage. Months unemployed, distance to a training centre
and the length of the caseworker's note are simulated pre-enrolment columns.
Enrolment, the certificate, survey response, hours and follow-up earnings are
simulated from a known structural model, so every effect has a true value.

The model has five features an analysis has to handle.

- **Confounding.** Enrolment falls with prior earnings, which also drive
  follow-up earnings. The raw enrolled-minus-not difference in earnings is
  about −13,300 dollars, while the true average effect is about +4,300.
- **Post-enrolment variables.** The certificate and hours worked lie on the
  path from enrolment to earnings, and survey response depends on enrolment.
  Adjusting for any of them changes the question.
- **Thin overlap.** Above 40,000 dollars of prior earnings, enrolment
  probability is below 0.03; in the dev data 29 of 3,223 such applicants
  enrolled. Effects there are extrapolations.
- **Two channels.** Enrolment raises the probability of employment, most
  for applicants below about 15,000 dollars of prior earnings, and raises the
  earnings of the employed, most for applicants with less education and for
  younger applicants on the log scale.
- **Dollar-scale heterogeneity.** Because the earnings effect is
  proportional, the effect in dollars is larger for applicants with higher
  baseline earnings. On the dollar scale the material effect modifiers are
  prior earnings, education, months unemployed and part-time work.

Each analyst saw one replicate (dev). The grader re-ran every pipeline,
unattended, on ten hidden replicates: six with new draws of applicants,
enrolment and noise (H1–H6), two in which enrolment has no effect but the
same confounding remains (N1–N2), and two with 1,500 training rows (S1–S2).
The 4,000 new applicants to be ranked over-represent low earners. Every run
used 4 CPU threads and a 10-minute limit; a failed or timed-out run counts as
making no claims.

### The answer key

The true values on the dev replicate, and the role of every column in the
generating model:

In [3]:
dev_truth = json.loads((TRUTH / "reps" / "dev" / "truth.json").read_text())
qt = pl.DataFrame([{"query": q, "truth": dev_truth[q]} for q in grade.QUERIES])
GT(qt).tab_header(title="True average effects on the dev replicate",
                  subtitle="Dollars of follow-up earnings (EMP_ATE: probability of employment)") \
    .fmt_number(["truth"], decimals=0, rows=list(range(7))).fmt_number(["truth"], decimals=3, rows=[7]) \
    .cols_label(query="Query", truth="True value") \
    .tab_source_note("Raw difference in mean observed earnings, enrolled minus not enrolled, on dev: "
                     "about -13,300 dollars.")

GT(_tbl_data=shape: (8, 2)
┌────────────────┬─────────────┐
│ query          ┆ truth       │
│ ---            ┆ ---         │
│ str            ┆ f64         │
╞════════════════╪═════════════╡
│ ATE            ┆ 4272.325552 │
│ ATT            ┆ 3369.905659 │
│ CATE_educ_lt12 ┆ 4235.121668 │
│ CATE_educ_ge16 ┆ 5683.125556 │
│ CATE_prior_q1  ┆ 3266.135293 │
│ CATE_prior_q4  ┆ 6706.947479 │
│ CATE_age_lt25  ┆ 3673.209931 │
│ EMP_ATE        ┆ 0.072453    │
└────────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x10a302de0>, _boxhead=Boxhead([ColInfo(var='query', type=<ColInfoTypeEnum.default: 1>, column_label='Query', column_align='left', column_width=None), ColInfo(var='truth', type=<ColInfoTypeEnum.default: 1>, column_label='True value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10bbd4d70>, _spanners=Spanners([]), _heading=Heading(title='True average effects on the dev replicate', subtitle='Dollars of follow-up earnings (EMP_ATE: probability of employment)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10bbd4a70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10bbd7140>, _source_notes=['Raw difference in mean observed earnings, enrolled minus not enrolled, on dev: about -13,300 dollars.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10bbd73b0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10bbd55b0>, <great_tables._gt_data.FormatInfo object at 0x10bbd74a0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, c

In [4]:
roles = pl.DataFrame([{"column": c, "role": r} for c, r in KEY["roles"].items()])
GT(roles).tab_header(title="Causal roles in the generating model (hidden from the analysts)") \
    .cols_label(column="Column", role="Role") \
    .tab_source_note("Material effect modifiers on the dollar scale: " + ", ".join(KEY["effect_modifiers"]) + ".")

GT(_tbl_data=shape: (16, 2)
┌─────────────────────┬──────────────────────────┐
│ column              ┆ role                     │
│ ---                 ┆ ---                      │
│ str                 ┆ str                      │
╞═════════════════════╪══════════════════════════╡
│ prior_earnings      ┆ confounder               │
│ education           ┆ confounder               │
│ months_unemployed   ┆ confounder               │
│ experience          ┆ confounder               │
│ age                 ┆ confounder               │
│ …                   ┆ …                        │
│ hours_post          ┆ mediator                 │
│ responded           ┆ post_treatment_selection │
│ caseworker_note_len ┆ irrelevant               │
│ enrolled            ┆ treatment                │
│ earnings_post       ┆ outcome                  │
└─────────────────────┴──────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10baa38c0>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='role', type=<ColInfoTypeEnum.default: 1>, column_label='Role', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10bbd5d90>, _spanners=Spanners([]), _heading=Heading(title='Causal roles in the generating model (hidden from the analysts)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a325790>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10a324e90>, _source_notes=['Material effect modifiers on the dollar scale: prior_earnings, education, months_unemployed, parttime.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10a325d90>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=T

## Guidance given to the analysts

Each analyst was a Claude Haiku agent, launched once with the assignment
below and the brief that follows it, with no further help on the analysis.
The brief and the assignment are reproduced in full; the departures section
lists the procedural messages sent during the runs.

In [5]:
display(Markdown((ROOT / "ASSIGNMENT.md").read_text()))

# Analyst assignment

Each analyst is a Claude Haiku agent launched once, in the background, with
the message below and no further help. Only the library name and version,
the run directory, the extra text and the documentation source change
between analysts. All analysts also receive `BRIEF.md` word for word.

## Message template

> Assignment: you are the analyst for the library **{LIBRARY}** (Python package `{PACKAGE}`, version {VERSION}{EXTRA}).
>
> Read the brief at docs/book/benchmarks/causal_discovery/BRIEF.md and carry it out completely. Your run directory is docs/book/benchmarks/causal_discovery/{RUN_DIR}/ — write every deliverable there. The data is in docs/book/benchmarks/causal_discovery/data/dev/.
>
> Documentation you may use for {LIBRARY}: {DOCS}. Do not read anything else in the repository, and do not read any other analyst's run directory.
>
> Work on your own, without asking questions; nobody will answer. Deliver all eight items in the brief. Before you finish, confirm that `python run.py --train <dev train> --test <dev test> --out dev_out --seed 1` runs from your run directory and writes findings.json and cate_predictions.parquet, and that refit.py imports and runs. Your final message: a short list of the deliverables written, the final model, and anything you could not complete. Then put the full text of REPORT.md and FRICTION.md in your final message as well, each between lines `----- REPORT.md -----` / `----- end REPORT.md -----` (and likewise for FRICTION.md), because the file tool may refuse to write report files.

## Per-library values

| Library | Package, version | Extra text | Documentation allowed |
|---|---|---|---|
| mboostrs | `mboostrs` 0.1.3 | "; it ships an agent guide: `python -m mboostrs guide` and `python -m mboostrs guide <topic>`" | `help()`, the agent guide, installed source (`python/mboostrs/`), and the book at `docs/book/` (usage/, methodology/, appendix/, case_studies/); docs/book/benchmarks/ and docs/book/cases/ excluded |
| EconML | `econml` 0.17.0 | "; scikit-learn estimators may be used as nuisance models" | `help()`, installed source, https://econml.azurewebsites.net/ |
| DoubleML | `doubleml` 0.11.4 | "; scikit-learn estimators may be used as nuisance learners" | `help()`, installed source, https://docs.doubleml.org/ |
| statsmodels | `statsmodels` 0.15.0 | "; any of its regression, GLM, GAM, discrete-choice or treatment-effect tools, with patsy formulas if you like" | `help()`, installed source, https://www.statsmodels.org/stable/ |
| LightGBM | `lightgbm` 4.7.0 | "; build any meta-learner you need from LightGBM models" | `help()`, installed source, https://lightgbm.readthedocs.io/ |

Run directories: `runs/<library>/` for the first run and `rerun/mboostrs/`
for the mboostrs rerun after the fixes.


In [6]:
display(Markdown((ROOT / "BRIEF.md").read_text()))

# Analyst brief: did the job-training program raise earnings, and for whom?

You are the analyst for one modelling library (named in your assignment).
Several analysts receive this same brief, each with a different library, and
work independently. Do the best causal analysis your library allows, as a
careful applied statistician would.

## Setting

A state workforce agency ran a voluntary job-training program. Applicants met a
caseworker, and each decided whether to enrol. Enrolment was not randomized.
Caseworkers steered applicants with low prior earnings towards the program, and
applicants earning more than 40,000 dollars a year were rarely eligible. A
follow-up survey one year later recorded earnings; not every applicant answered
it. The agency wants to know what the program did and whom to enrol next year,
when the budget covers 20% of the new applicants.

## Data

`data/dev/train.parquet` (12,000 applicants from the program year) and
`data/dev/test.parquet` (4,000 new applicants for next year, pre-enrolment
columns only). The applicant characteristics come from real survey records of
US men; the program, the survey response and the follow-up outcomes are
simulated from a process that is hidden from you. Treat it as a real
evaluation. The new applicants are not a random sample of the program-year
applicants.

| Column | Measured | Meaning |
|---|---|---|
| `row_id` | — | Row identifier (not a covariate) |
| `age` | at application | Years |
| `education` | at application | Years of schooling |
| `experience` | at application | Years of potential work experience (age − education − 6) |
| `ethnicity` | at application | `cauc` or `afam` |
| `smsa` | at application | Lives in a metropolitan area (`yes`/`no`) |
| `region` | at application | `northeast`, `midwest`, `south`, `west` |
| `parttime` | at application | Worked part time in the year before applying (`yes`/`no`) |
| `prior_earnings` | at application | Earnings in the year before applying, dollars |
| `months_unemployed` | at application | Months without work in the two years before applying |
| `distance_km` | at application | Distance from home to the nearest training centre, km |
| `caseworker_note_len` | at application | Length of the caseworker's intake note, characters |
| `enrolled` | at enrolment | 1 if the applicant enrolled in the program (train only) |
| `certificate` | during the program | 1 if the applicant earned the program certificate (train only) |
| `responded` | at follow-up | 1 if the applicant answered the follow-up survey (train only) |
| `hours_post` | at follow-up | Hours worked in the follow-up year; missing if no response (train only) |
| `earnings_post` | at follow-up | Earnings in the follow-up year, dollars; 0 if not employed; missing if no response (train only) |

## Questions from the agency

1. What was the average effect of enrolling on follow-up earnings, in dollars,
   over all program-year applicants (ATE) and over those who enrolled (ATT)?
2. Did the program work by getting people into work, by raising the earnings
   of those in work, or both?
3. For whom was the effect larger or smaller? Name the characteristics that
   change the size of the effect, and describe how.
4. Which columns must an analysis adjust for, and which must it not adjust
   for? Give a role for every column from `age` to `earnings_post`.
5. For which applicants can these data not tell what the program did?
6. Rank next year's applicants so that enrolling the top 20% gives the largest
   total gain in earnings.
7. How sensitive are your conclusions to a characteristic that affects both
   enrolment and earnings but is not in the data?

## Effect queries

Each query is an average over the 12,000 program-year applicants in the
training file (all of them, whether or not they answered the survey), of the
difference between expected follow-up earnings if enrolled and if not
enrolled. Give a point estimate and a 90% interval.

| Id | Population |
|---|---|
| ATE | All applicants |
| ATT | Applicants who enrolled |
| CATE_educ_lt12 | `education` < 12 |
| CATE_educ_ge16 | `education` ≥ 16 |
| CATE_prior_q1 | `prior_earnings` at or below its 25th percentile in the training file |
| CATE_prior_q4 | `prior_earnings` at or above its 75th percentile in the training file |
| CATE_age_lt25 | `age` < 25 |
| EMP_ATE | All applicants; the outcome is employment (`earnings_post` > 0), so the effect is a difference in probability |

## Deliverables (all in your run directory)

1. **`run.py`**, the whole pipeline, run as
   `python run.py --train TRAIN --test TEST --out OUTDIR --seed SEED`.
   It reads only the two files it is given, makes every modelling decision
   from that data, and writes `OUTDIR/findings.json` and
   `OUTDIR/cate_predictions.parquet`. It will be re-run, unattended, on other
   draws of this problem: some with different noise, some with only 1,500
   training rows, and some where enrolment has no effect at all. Do not
   hard-code findings. It must finish within 10 minutes with 4 CPU threads.
2. **`findings.json`**, schema below.
3. **`cate_predictions.parquet`**: columns `row_id`, `cate`, `lower90`,
   `upper90`, one row per test applicant: the expected effect of enrolling
   that applicant on follow-up earnings, in dollars, with a 90% interval.
   The ranking for question 6 is the order of `cate`.
4. **`refit.py`**, defining
   `fit_ate(train: polars.DataFrame, drop: list[str]) -> float`, which fits
   your final configuration (no tuning) without the columns in `drop` and
   returns the ATE estimate. It is used to time follow-up analyses.
5. **`REPORT.md`**, a report for the agency, at most 900 words, from your
   dev-data run: answers to questions 1–7, how sure you are of each, what the
   data cannot tell, and what you would check next.
6. **`tuning_log.csv`**, one row per configuration you tried, with its
   settings, the validation score and whether it was kept.
7. **`FRICTION.md`**: every point where the library made the task harder.
   Quote each library error or warning message you met verbatim, and say
   whether the message told you how to fix the problem.
8. Run `run.py` on the dev data with `--seed 1 --out dev_out` and keep the
   outputs.

### `findings.json` schema

```json
{
  "library": "name and version",
  "effect_queries": {
    "ATE": {"estimate": 1000.0, "lower90": 500.0, "upper90": 1500.0}
  },
  "roles": {"age": "confounder"},
  "adjustment_set": ["age"],
  "effect_modifiers": [
    {"variable": "education", "direction": "larger_when_lower", "evidence": "one line"}
  ],
  "channels": {"employment": true, "earnings_if_employed": true},
  "limited_overlap": [
    {"description": "who the data cannot speak for", "variable": "prior_earnings"}
  ],
  "warnings": ["anything the pipeline found that a reader should know"]
}
```

- Every query must appear. Use `null` for an interval bound you cannot produce.
- `roles` has one entry for every column from `age` to `earnings_post`, each
  one of `confounder`, `instrument`, `treatment_only`, `outcome_only`,
  `mediator`, `post_treatment_selection`, `treatment`, `outcome`,
  `irrelevant`.
- `effect_modifiers` lists only characteristics you conclude change the
  effect in dollars; `direction` is `larger_when_higher`,
  `larger_when_lower` or `nonmonotone`.
- `channels` says which channels you conclude carry a non-zero part of the
  effect.

## Rules

- Use the library you are assigned for the modelling. NumPy, Polars, pandas
  and SciPy may be used for data handling and for simple arithmetic around the
  model; another modelling library may not.
- Documentation: the library's docstrings (`help()`), its installed source
  code, and its official documentation. Do not read anything else in this
  repository, apart from the paths named in your assignment.
- Budget: at most 40 configurations in `tuning_log.csv`.
- Python: `source .venv/bin/activate`.
  Other analysts share the machine: export `OMP_NUM_THREADS=2` and
  `RAYON_NUM_THREADS=2` while developing, and do not start more than one
  long-running process at a time.
- Graders will score: the effect estimates and the coverage of their
  intervals (including on draws where the program has no effect), the
  per-applicant effects and their intervals, the value of your top-20%
  ranking, the variable roles, the effect modifiers and channels you claim
  (and the absence of false claims), the report, reproducibility of `run.py`,
  and the time taken by follow-up analyses.


## Average effects

Each pipeline reports eight averages over the 12,000 program-year applicants:
the ATE, the ATT, five subgroup effects and the effect on employment. A query
is covered when the 90% interval contains the true value.

In [7]:
qrows = []
for lib in LIBS:
    s = summary[lib]
    for q in grade.QUERIES:
        widths = [per[lib][r]["queries"][q]["width"] for r in ALT if per[lib][r]["queries"][q]["width"] is not None]
        qrows.append({"query": q, "library": LIBS[lib], "coverage": s["coverage"][q],
                      "rmse": s["rmse"][q], "width": float(np.mean(widths)) if widths else None})
qd = pl.DataFrame(qrows).pivot(on="library", index="query", values=["coverage", "rmse"])
GT(qd, rowname_col="query").tab_header(
    title="Effect queries: 90% interval coverage and RMSE by query",
    subtitle="8 replicates with effects (H1-H6, S1-S2); RMSE in dollars except EMP_ATE (probability)") \
    .fmt_number([c for c in qd.columns if c.startswith("coverage")], decimals=2) \
    .fmt_number([c for c in qd.columns if c.startswith("rmse")], decimals=0, rows=list(range(7))) \
    .fmt_number([c for c in qd.columns if c.startswith("rmse")], decimals=3, rows=[7]) \
    .sub_missing(missing_text="none")

GT(_tbl_data=shape: (8, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ query     ┆ coverage_ ┆ coverage_ ┆ coverage_ ┆ … ┆ rmse_Econ ┆ rmse_Doub ┆ rmse_stat ┆ rmse_Lig │
│ ---       ┆ mboostrs  ┆ EconML    ┆ DoubleML  ┆   ┆ ML 0.17.0 ┆ leML      ┆ smodels   ┆ htGBM    │
│ str       ┆ 0.1.3     ┆ 0.17.0    ┆ 0.11.4    ┆   ┆ ---       ┆ 0.11.4    ┆ 0.15.0    ┆ 4.7.0    │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ f64       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆           ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ ATE       ┆ 0.875     ┆ 0.875     ┆ 0.75      ┆ … ┆ 1157.3821 ┆ 2033.1051 ┆ 593.33217 ┆ 2707.973 │
│           ┆           ┆           ┆           ┆   ┆ 07        ┆ 64        ┆ 6         ┆ 718      │
│ ATT       ┆ 1.0       ┆ 1.0       ┆ 1.0       ┆ … ┆ 365.89319 ┆ 188.53394 ┆ 503.14153 ┆ 361.7443 │
│           ┆           ┆           ┆           ┆   ┆ 6         ┆ 5         ┆ 1         ┆ 77       │
│ CATE_educ ┆ 0.875     ┆ 0.625     ┆ 1.0       ┆ … ┆ 1133.2884 ┆ 983.82491 ┆ 908.96789 ┆ 1518.921 │
│ _lt12     ┆           ┆           ┆           ┆   ┆ 81        ┆ 4         ┆ 2         ┆ 161      │
│ CATE_educ ┆ 0.875     ┆ 0.875     ┆ 0.625     ┆ … ┆ 2151.7926 ┆ 6551.2960 ┆ 1775.6637 ┆ 8538.732 │
│ _ge16     ┆           ┆           ┆           ┆   ┆ 15        ┆ 49        ┆ 15        ┆ 383      │
│ CATE_prio ┆ 1.0       ┆ 0.75      ┆ 1.0       ┆ … ┆ 558.78208 ┆ 553.93642 ┆ 814.87357 ┆ 214.0604 │
│ r_q1      ┆           ┆           ┆           ┆   ┆ 1         ┆ 9         ┆ 2         ┆ 09       │
│ CATE_prio ┆ 0.875     ┆ 0.875     ┆ 0.875     ┆ … ┆ 3971.6318 ┆ 7759.3686 ┆ 2786.7866 ┆ 10112.81 │
│ r_q4      ┆           ┆           ┆           ┆   ┆ 22        ┆ 4         ┆ 34        ┆ 0267     │
│ CATE_age_ ┆ 0.875     ┆ 1.0       ┆ 0.875     ┆ … ┆ 531.64829 ┆ 779.23295 ┆ 680.73360 ┆ 1006.055 │
│ lt25      ┆           ┆           ┆           ┆   ┆ 8         ┆ 9         ┆ 8         ┆ 2        │
│ EMP_ATE   ┆ 0.875     ┆ 0.125     ┆ 0.875     ┆ … ┆ 0.042204  ┆ 0.020641  ┆ 0.016975  ┆ 0.021065 │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10bbd6750>, _boxhead=Boxhead([ColInfo(var='query', type=<ColInfoTypeEnum.stub: 2>, column_label='query', column_align='left', column_width=None), ColInfo(var='coverage_mboostrs 0.1.3', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_mboostrs 0.1.3', column_align='right', column_width=None), ColInfo(var='coverage_EconML 0.17.0', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_EconML 0.17.0', column_align='right', column_width=None), ColInfo(var='coverage_DoubleML 0.11.4', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_DoubleML 0.11.4', column_align='right', column_width=None), ColInfo(var='coverage_statsmodels 0.15.0', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_statsmodels 0.15.0', column_align='right', column_width=None), ColInfo(var='coverage_LightGBM 4.7.0', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_LightGBM 4.7.0', column_align='right', column_width=None), ColInfo(var='rmse_mboostrs 0.1.3', type=<ColInfoTypeEnum.default: 1>, column_label='rmse_mboostrs 0.1.3', column_align='right', column_width=None), ColInfo(var='rmse_EconML 0.17.0', type=<ColInfoTypeEnum.default: 1>, column_label='rmse_EconML 0.17.0', column_align='right', column_width=None), ColInfo(var='rmse_DoubleML 0.11.4', type=<ColInfoTypeEnum.default: 1>, column_label='rmse_DoubleML 0.11.4', column_align='right', column_width=None), ColInfo(var='rmse_statsmodels 0.15.0', type=<ColInfoTypeEnum.default: 1>, column_label='rmse_statsmodels 0.15.0', column_align='right', column_width=None), ColInfo(var='rmse_LightGBM 4.7.0', type=<ColInfoTypeEnum.default: 1>

The top quartile of prior earnings is where the queries fail: its enrolment
probability is below 0.03, and every library's RMSE there is several times its
RMSE elsewhere. The mboostrs ATE intervals are wide enough to cover the truth
on 7 of 8 replicates, but its point estimates spread widely: the pipeline adds
a step covariate at a data-chosen eligibility cut and a linear outcome
baseline, and the estimate moves with where the cut falls. The analyst's
report gives the ATE as "roughly 3,000 to 6,000", and its 200,000-row
follow-up fit returned 7,988 dollars against a truth of about 4,300.

### No-effect replicates

In N1 and N2 enrolment has no effect, but the enrolment model and the
confounding are unchanged, so the raw enrolled-minus-not difference stays
large and negative. A false claim is a query interval that excludes zero, an
effect modifier, or a channel.

In [8]:
nrows = []
for lib in LIBS:
    for rep in ["N1", "N2"]:
        r = per[lib][rep]
        nrows.append({"library": LIBS[lib], "replicate": rep, "ran": r["ran"],
                      "intervals_excluding_zero": sum(v["excludes_zero"] for v in r["queries"].values()),
                      "modifiers": ", ".join(r["modifiers"]["claimed"]) or "none",
                      "channels_claimed": 2 - r["channels_correct"],
                      "ate": r["queries"]["ATE"]["estimate"]})
GT(pl.DataFrame(nrows), rowname_col="library", groupname_col="replicate").tab_header(
    title="No-effect replicates: what each pipeline claimed",
    subtitle="Enrolment has no effect; the raw enrolled-minus-not difference is still large and negative") \
    .fmt_integer(["ate"]).cols_label(ran="Ran", intervals_excluding_zero="Query intervals excluding 0",
                                     modifiers="Modifiers claimed", channels_claimed="Channels claimed",
                                     ate="ATE estimate ($)")

GT(_tbl_data=shape: (10, 7)
┌────────────────┬───────────┬──────┬───────────────┬───────────────┬───────────────┬──────────────┐
│ library        ┆ replicate ┆ ran  ┆ intervals_exc ┆ modifiers     ┆ channels_clai ┆ ate          │
│ ---            ┆ ---       ┆ ---  ┆ luding_zero   ┆ ---           ┆ med           ┆ ---          │
│ str            ┆ str       ┆ bool ┆ ---           ┆ str           ┆ ---           ┆ f64          │
│                ┆           ┆      ┆ i64           ┆               ┆ i64           ┆              │
╞════════════════╪═══════════╪══════╪═══════════════╪═══════════════╪═══════════════╪══════════════╡
│ mboostrs 0.1.3 ┆ N1        ┆ true ┆ 0             ┆ none          ┆ 0             ┆ 481.800104   │
│ mboostrs 0.1.3 ┆ N2        ┆ true ┆ 0             ┆ none          ┆ 0             ┆ 145.344854   │
│ EconML 0.17.0  ┆ N1        ┆ true ┆ 0             ┆ none          ┆ 0             ┆ -147.138429  │
│ EconML 0.17.0  ┆ N2        ┆ true ┆ 0             ┆ none          ┆ 0             ┆ 203.462389   │
│ DoubleML       ┆ N1        ┆ true ┆ 0             ┆ none          ┆ 1             ┆ -1347.28345  │
│ 0.11.4         ┆           ┆      ┆               ┆               ┆               ┆              │
│ DoubleML       ┆ N2        ┆ true ┆ 4             ┆ none          ┆ 1             ┆ -1405.234276 │
│ 0.11.4         ┆           ┆      ┆               ┆               ┆               ┆              │
│ statsmodels    ┆ N1        ┆ true ┆ 0             ┆ none          ┆ 0             ┆ 115.194216   │
│ 0.15.0         ┆           ┆      ┆               ┆               ┆               ┆              │
│ statsmodels    ┆ N2        ┆ true ┆ 0             ┆ none          ┆ 0             ┆ -6.579659    │
│ 0.15.0         ┆           ┆      ┆               ┆               ┆               ┆              │
│ LightGBM 4.7.0 ┆ N1        ┆ true ┆ 2             ┆ education     ┆ 2             ┆ -2091.048197 │
│ LightGBM 4.7.0 ┆ N2        ┆ true ┆ 3             ┆ prior_earning ┆ 1             ┆ -1934.719249 │
│                ┆           ┆      ┆               ┆ s             ┆               ┆              │
└────────────────┴───────────┴──────┴───────────────┴───────────────┴───────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x10bbd5ac0>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='replicate', type=<ColInfoTypeEnum.row_group: 3>, column_label='replicate', column_align='left', column_width=None), ColInfo(var='ran', type=<ColInfoTypeEnum.default: 1>, column_label='Ran', column_align='center', column_width=None), ColInfo(var='intervals_excluding_zero', type=<ColInfoTypeEnum.default: 1>, column_label='Query intervals excluding 0', column_align='right', column_width=None), ColInfo(var='modifiers', type=<ColInfoTypeEnum.default: 1>, column_label='Modifiers claimed', column_align='left', column_width=None), ColInfo(var='channels_claimed', type=<ColInfoTypeEnum.default: 1>, column_label='Channels claimed', column_align='right', column_width=None), ColInfo(var='ate', type=<ColInfoTypeEnum.default: 1>, column_label='ATE estimate ($)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10bb6e3c0>, _spanners=Spanners([]), _heading=Heading(title='No-effect replicates: what each pipeline claimed', subtitle='Enrolment has no effect; the raw enrolled-minus-not difference is still large and negative', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10bc22bd0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10bc21580>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10bc22c90>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10bb6e360>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table',

## Per-applicant effects and the enrolment ranking

Each pipeline predicts the effect of enrolling each of 4,000 new applicants,
with a 90% interval; the top 20% by predicted effect is the enrolment choice.
Regret measures how much of the gain from the best possible choice the ranking
gives up: 0 is the best 20%, 1 is a random 20%.

In [9]:
crows = []
for lib in LIBS:
    for r in ALT:
        c = per[lib][r].get("cate", {})
        if "pehe" in c:
            crows.append({"library": LIBS[lib], "replicate": r, "pehe": c["pehe"], "coverage": c["coverage"],
                          "width": c["mean_width"], "chosen": c["policy_value"], "oracle": c["policy_oracle"],
                          "random": c["policy_random"]})
cd = pl.DataFrame(crows).group_by("library", maintain_order=True).agg(pl.all().exclude("replicate").mean())
GT(cd, rowname_col="library").tab_header(
    title="Per-applicant effects and the top-20% enrolment policy",
    subtitle="Means over H1-H6, S1-S2; dollars per applicant; 4,000 new applicants per replicate") \
    .fmt_integer(["pehe", "width", "chosen", "oracle", "random"]).fmt_number(["coverage"], decimals=2) \
    .cols_label(pehe="PEHE", coverage="90% coverage", width="Mean interval width",
                chosen="Mean effect, chosen 20%", oracle="Mean effect, best 20%", random="Mean effect, all")

GT(_tbl_data=shape: (5, 7)
┌───────────────┬─────────────┬───────────┬──────────────┬─────────────┬─────────────┬─────────────┐
│ library       ┆ pehe        ┆ coverage  ┆ width        ┆ chosen      ┆ oracle      ┆ random      │
│ ---           ┆ ---         ┆ ---       ┆ ---          ┆ ---         ┆ ---         ┆ ---         │
│ str           ┆ f64         ┆ f64       ┆ f64          ┆ f64         ┆ f64         ┆ f64         │
╞═══════════════╪═════════════╪═══════════╪══════════════╪═════════════╪═════════════╪═════════════╡
│ mboostrs      ┆ 2867.459819 ┆ 0.661469  ┆ 6776.845386  ┆ 4423.722544 ┆ 6329.109871 ┆ 3858.396697 │
│ 0.1.3         ┆             ┆           ┆              ┆             ┆             ┆             │
│ EconML 0.17.0 ┆ 2073.087167 ┆ 0.825844  ┆ 5018.074639  ┆ 5444.404265 ┆ 6329.109871 ┆ 3858.396697 │
│ DoubleML      ┆ 5478.167172 ┆ 0.876531  ┆ 14786.232254 ┆ 4418.669207 ┆ 6329.109871 ┆ 3858.396697 │
│ 0.11.4        ┆             ┆           ┆              ┆             ┆             ┆             │
│ statsmodels   ┆ 1985.34713  ┆ 0.8906875 ┆ 5646.192812  ┆ 4259.819401 ┆ 6329.109871 ┆ 3858.396697 │
│ 0.15.0        ┆             ┆           ┆              ┆             ┆             ┆             │
│ LightGBM      ┆ 8251.110919 ┆ 0.808219  ┆ 10932.585305 ┆ 4522.108298 ┆ 6329.109871 ┆ 3858.396697 │
│ 4.7.0         ┆             ┆           ┆              ┆             ┆             ┆             │
└───────────────┴─────────────┴───────────┴──────────────┴─────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x10bc232c0>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='pehe', type=<ColInfoTypeEnum.default: 1>, column_label='PEHE', column_align='right', column_width=None), ColInfo(var='coverage', type=<ColInfoTypeEnum.default: 1>, column_label='90% coverage', column_align='right', column_width=None), ColInfo(var='width', type=<ColInfoTypeEnum.default: 1>, column_label='Mean interval width', column_align='right', column_width=None), ColInfo(var='chosen', type=<ColInfoTypeEnum.default: 1>, column_label='Mean effect, chosen 20%', column_align='right', column_width=None), ColInfo(var='oracle', type=<ColInfoTypeEnum.default: 1>, column_label='Mean effect, best 20%', column_align='right', column_width=None), ColInfo(var='random', type=<ColInfoTypeEnum.default: 1>, column_label='Mean effect, all', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10bbd6690>, _spanners=Spanners([]), _heading=Heading(title='Per-applicant effects and the top-20% enrolment policy', subtitle='Means over H1-H6, S1-S2; dollars per applicant; 4,000 new applicants per replicate', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10bc23890>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10bc23140>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10bc23830>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10bc23770>, <great_tables._gt_data.FormatInfo object at 0x10bc238f0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, cate

Most rankings are close to random because they follow the extrapolated
effects of high earners, whose effects are the largest in dollars and the
least supported by the data. EconML's ranking is the best because its
effect model is linear in prior earnings. The mboostrs pipeline ranks only
applicants inside the overlap region (`rank_by_effect`), but its effect model
has education as the only modifier, so its ranking chooses among applicants
with 16 or more years of schooling and breaks ties by row order; its regret
(0.77) is close to the others'. Its per-applicant intervals come from a REML
refit of the second stage and do not include the error of the nuisance
models.

## Causal structure

The roles each pipeline gave on replicate H1, and whether its adjustment set
contains every confounder group and no post-enrolment column. Every pipeline
left the certificate, hours and survey response out of the adjustment, and
every pipeline named distance as affecting enrolment only. The common errors
are calling education an outcome-only column (it also changes enrolment) and
leaving metropolitan residence out of the adjustment set (it shortens the
distance to a centre and raises earnings).

In [10]:
cols = list(KEY["roles"])
rrows = []
for lib in LIBS:
    r = per[lib]["H1"]["roles"]
    rrows.append({"library": LIBS[lib], **{c: r["correct"][c] for c in cols},
                  "adjusts_post": ", ".join(r["adjusts_post_treatment"]) or "none",
                  "missing": "; ".join("/".join(g) for g in r["missing_confounders"]) or "none"})
rt = pl.DataFrame(rrows)
gt = GT(rt, rowname_col="library").tab_header(
    title="Roles given on replicate H1 (true = matches the key)",
    subtitle="Distance and ethnicity count as correct under instrument or treatment_only") \
    .cols_label(adjusts_post="Post-enrolment columns adjusted for", missing="Confounder groups missing from the adjustment set")
gt

GT(_tbl_data=shape: (5, 19)
┌───────────┬───────────┬───────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ library   ┆ prior_ear ┆ education ┆ months_un ┆ … ┆ enrolled ┆ earnings_ ┆ adjusts_p ┆ missing   │
│ ---       ┆ nings     ┆ ---       ┆ employed  ┆   ┆ ---      ┆ post      ┆ ost       ┆ ---       │
│ str       ┆ ---       ┆ bool      ┆ ---       ┆   ┆ bool     ┆ ---       ┆ ---       ┆ str       │
│           ┆ bool      ┆           ┆ bool      ┆   ┆          ┆ bool      ┆ str       ┆           │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ mboostrs  ┆ true      ┆ true      ┆ true      ┆ … ┆ true     ┆ true      ┆ none      ┆ none      │
│ 0.1.3     ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ EconML    ┆ true      ┆ true      ┆ true      ┆ … ┆ true     ┆ true      ┆ none      ┆ none      │
│ 0.17.0    ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ DoubleML  ┆ true      ┆ false     ┆ true      ┆ … ┆ true     ┆ true      ┆ none      ┆ none      │
│ 0.11.4    ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ statsmode ┆ true      ┆ false     ┆ true      ┆ … ┆ true     ┆ true      ┆ none      ┆ education │
│ ls 0.15.0 ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆ ; age/exp │
│           ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆ erience;  │
│           ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆ sms…      │
│ LightGBM  ┆ true      ┆ false     ┆ true      ┆ … ┆ true     ┆ true      ┆ none      ┆ region;   │
│ 4.7.0     ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆ smsa      │
└───────────┴───────────┴───────────┴───────────┴───┴──────────┴───────────┴───────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x10b273fb0>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='prior_earnings', type=<ColInfoTypeEnum.default: 1>, column_label='prior_earnings', column_align='center', column_width=None), ColInfo(var='education', type=<ColInfoTypeEnum.default: 1>, column_label='education', column_align='center', column_width=None), ColInfo(var='months_unemployed', type=<ColInfoTypeEnum.default: 1>, column_label='months_unemployed', column_align='center', column_width=None), ColInfo(var='experience', type=<ColInfoTypeEnum.default: 1>, column_label='experience', column_align='center', column_width=None), ColInfo(var='age', type=<ColInfoTypeEnum.default: 1>, column_label='age', column_align='center', column_width=None), ColInfo(var='region', type=<ColInfoTypeEnum.default: 1>, column_label='region', column_align='center', column_width=None), ColInfo(var='smsa', type=<ColInfoTypeEnum.default: 1>, column_label='smsa', column_align='center', column_width=None), ColInfo(var='ethnicity', type=<ColInfoTypeEnum.default: 1>, column_label='ethnicity', column_align='center', column_width=None), ColInfo(var='distance_km', type=<ColInfoTypeEnum.default: 1>, column_label='distance_km', column_align='center', column_width=None), ColInfo(var='parttime', type=<ColInfoTypeEnum.default: 1>, column_label='parttime', column_align='center', column_width=None), ColInfo(var='certificate', type=<ColInfoTypeEnum.default: 1>, column_label='certificate', column_align='center', column_width=None), ColInfo(var='hours_post', type=<ColInfoTypeEnum.default: 1>, column_label='hours_post', column_align='center', column_width=None), ColInfo(var='responded', type=<ColInfoTypeEnum.default: 1>, column_label='responded', column_align='center', column_width=None), ColInfo(var='caseworker_note_len', type=<ColInfoTypeEnum.default: 1>, column_label='caseworker_note_len', column_align='center', column_width=None), ColInfo(var='enrolled'

## Run time and reproducibility

Pipelines ran on a shared machine; the load average was between 10 and 40
during the evaluations, so the times are upper bounds for an idle machine.

In [11]:
srows = []
for lib in LIBS:
    rl = runlogs[lib]
    times = rl.get("followup", {}).get("times") or []
    run_secs = [rl[r]["seconds"] for r in ["dev"] + HIDDEN if r in rl and rl[r].get("ok")]
    srows.append({"library": LIBS[lib], "run_median": float(np.median(run_secs)) if run_secs else None,
                  "followup_median": float(np.median(times)) if times else None,
                  "followup_p90": float(np.quantile(times, 0.9)) if times else None,
                  "large_n": rl.get("large_n", {}).get("seconds") if rl.get("large_n", {}).get("ok") else None})
GT(pl.DataFrame(srows), rowname_col="library").tab_header(
    title="Run time", subtitle="Seconds, 4 threads, shared machine (load average 10 to 40 during the runs)") \
    .fmt_number(["run_median", "followup_median", "followup_p90", "large_n"], decimals=1) \
    .sub_missing(missing_text="failed") \
    .cols_label(run_median="Full pipeline, median over runs", followup_median="Follow-up fit_ate, median",
                followup_p90="Follow-up fit_ate, p90", large_n="fit_ate on 200,000 rows")

GT(_tbl_data=shape: (5, 5)
┌────────────────────┬────────────┬─────────────────┬──────────────┬────────────┐
│ library            ┆ run_median ┆ followup_median ┆ followup_p90 ┆ large_n    │
│ ---                ┆ ---        ┆ ---             ┆ ---          ┆ ---        │
│ str                ┆ f64        ┆ f64             ┆ f64          ┆ f64        │
╞════════════════════╪════════════╪═════════════════╪══════════════╪════════════╡
│ mboostrs 0.1.3     ┆ 92.967626  ┆ 8.168835        ┆ 8.729101     ┆ 357.62822  │
│ EconML 0.17.0      ┆ 11.056991  ┆ 1.101886        ┆ 1.156356     ┆ 7.386826   │
│ DoubleML 0.11.4    ┆ 97.247932  ┆ 11.097241       ┆ 12.820316    ┆ 326.416917 │
│ statsmodels 0.15.0 ┆ 26.725183  ┆ 0.076676        ┆ 0.081836     ┆ 1.788361   │
│ LightGBM 4.7.0     ┆ 367.716461 ┆ 7.106742        ┆ 9.48191      ┆ 31.348657  │
└────────────────────┴────────────┴─────────────────┴──────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x10bc22660>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='run_median', type=<ColInfoTypeEnum.default: 1>, column_label='Full pipeline, median over runs', column_align='right', column_width=None), ColInfo(var='followup_median', type=<ColInfoTypeEnum.default: 1>, column_label='Follow-up fit_ate, median', column_align='right', column_width=None), ColInfo(var='followup_p90', type=<ColInfoTypeEnum.default: 1>, column_label='Follow-up fit_ate, p90', column_align='right', column_width=None), ColInfo(var='large_n', type=<ColInfoTypeEnum.default: 1>, column_label='fit_ate on 200,000 rows', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10bc48c50>, _spanners=Spanners([]), _heading=Heading(title='Run time', subtitle='Seconds, 4 threads, shared machine (load average 10 to 40 during the runs)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x113168d40>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x113169100>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1131690a0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x113168f80>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x113168da0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(s

In [12]:
rp = []
for lib in LIBS:
    rl = runlogs[lib]
    g, a = rl.get("repro_grader_vs_grader", {}), rl.get("repro_grader_vs_analyst", {})
    rp.append({"library": LIBS[lib], "grader_twice_findings": str(g.get("findings_equal")),
               "grader_twice_max_diff": g.get("pred_max_abs_diff") if isinstance(g.get("pred_max_abs_diff"), float) else None,
               "vs_analyst_findings": str(a.get("findings_equal")),
               "vs_analyst_max_diff": a.get("pred_max_abs_diff") if isinstance(a.get("pred_max_abs_diff"), float) else None})
GT(pl.DataFrame(rp), rowname_col="library").tab_header(
    title="Reproducibility of the dev run", subtitle="Largest absolute difference in per-applicant effects, dollars") \
    .fmt_number(["grader_twice_max_diff", "vs_analyst_max_diff"], decimals=2) \
    .sub_missing(missing_text="n/a") \
    .cols_label(grader_twice_findings="Grader run twice: findings equal", grader_twice_max_diff="max diff",
                vs_analyst_findings="Grader vs analyst dev_out: findings equal", vs_analyst_max_diff="max diff")

GT(_tbl_data=shape: (5, 5)
┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┐
│ library           ┆ grader_twice_find ┆ grader_twice_max_ ┆ vs_analyst_findin ┆ vs_analyst_max_d │
│ ---               ┆ ings              ┆ diff              ┆ gs                ┆ iff              │
│ str               ┆ ---               ┆ ---               ┆ ---               ┆ ---              │
│                   ┆ str               ┆ f64               ┆ str               ┆ f64              │
╞═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪══════════════════╡
│ mboostrs 0.1.3    ┆ True              ┆ 0.0               ┆ True              ┆ 0.0              │
│ EconML 0.17.0     ┆ False             ┆ 0.0               ┆ False             ┆ 0.0              │
│ DoubleML 0.11.4   ┆ False             ┆ 3.6380e-12        ┆ False             ┆ 3.6380e-12       │
│ statsmodels       ┆ True              ┆ 0.0               ┆ True              ┆ 0.0              │
│ 0.15.0            ┆                   ┆                   ┆                   ┆                  │
│ LightGBM 4.7.0    ┆ False             ┆ 6308.503797       ┆ False             ┆ 3628.654569      │
└───────────────────┴───────────────────┴───────────────────┴───────────────────┴──────────────────┘, _body=<great_tables._gt_data.Body object at 0x10bbd6c00>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='grader_twice_findings', type=<ColInfoTypeEnum.default: 1>, column_label='Grader run twice: findings equal', column_align='left', column_width=None), ColInfo(var='grader_twice_max_diff', type=<ColInfoTypeEnum.default: 1>, column_label='max diff', column_align='right', column_width=None), ColInfo(var='vs_analyst_findings', type=<ColInfoTypeEnum.default: 1>, column_label='Grader vs analyst dev_out: findings equal', column_align='left', column_width=None), ColInfo(var='vs_analyst_max_diff', type=<ColInfoTypeEnum.default: 1>, column_label='max diff', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x113169160>, _spanners=Spanners([]), _heading=Heading(title='Reproducibility of the dev run', subtitle='Largest absolute difference in per-applicant effects, dollars', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10bc23d70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10bc22900>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x113169880>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10bc23590>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x113169610>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', t

The fourth mboostrs pipeline reproduced its findings exactly, both between
the grader's two runs and against the analyst's own dev run.

## Analyst effort

Tokens, tool calls and elapsed minutes per analyst. "Saved from message"
means the agent harness refused the report file and the text came back in the
analyst's final message; it was saved unchanged.

In [13]:
agents = pl.read_csv(ROOT / "agents.csv")
if VARIANT == "after":
    agents = agents.filter((pl.col("library") != "mboostrs") | (pl.col("run") == "rerun 3"))
else:
    agents = agents.filter((pl.col("library") != "mboostrs") | (pl.col("run") == "before"))
GT(agents.drop("run"), rowname_col="library").tab_header(
    title="What each analyst spent", subtitle="Claude Haiku agents, one per library") \
    .cols_label(tokens="Tokens", tool_calls="Tool calls", minutes="Elapsed minutes", report_file="REPORT.md",
                friction_file="FRICTION.md", procedural_messages="Procedural messages") \
    .fmt_integer(["tokens"]).fmt_number(["minutes"], decimals=0)

GT(_tbl_data=shape: (5, 7)
┌─────────────┬────────┬────────────┬─────────┬─────────────┬───────────────┬──────────────────────┐
│ library     ┆ tokens ┆ tool_calls ┆ minutes ┆ report_file ┆ friction_file ┆ procedural_messages  │
│ ---         ┆ ---    ┆ ---        ┆ ---     ┆ ---         ┆ ---           ┆ ---                  │
│ str         ┆ i64    ┆ i64        ┆ f64     ┆ str         ┆ str           ┆ i64                  │
╞═════════════╪════════╪════════════╪═════════╪═════════════╪═══════════════╪══════════════════════╡
│ econml      ┆ 238521 ┆ 67         ┆ 34.6    ┆ written     ┆ written       ┆ 0                    │
│ doubleml    ┆ 395416 ┆ 115        ┆ 89.3    ┆ saved from  ┆ saved from    ┆ 0                    │
│             ┆        ┆            ┆         ┆ message     ┆ message       ┆                      │
│ statsmodels ┆ 345173 ┆ 77         ┆ 23.7    ┆ written     ┆ written       ┆ 0                    │
│ lightgbm    ┆ 261680 ┆ 63         ┆ 24.8    ┆ written     ┆ written       ┆ 2                    │
│ mboostrs    ┆ 424446 ┆ 126        ┆ 86.1    ┆ saved from  ┆ written       ┆ 1                    │
│             ┆        ┆            ┆         ┆ message     ┆               ┆                      │
└─────────────┴────────┴────────────┴─────────┴─────────────┴───────────────┴──────────────────────┘, _body=<great_tables._gt_data.Body object at 0x11316a270>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='tokens', type=<ColInfoTypeEnum.default: 1>, column_label='Tokens', column_align='right', column_width=None), ColInfo(var='tool_calls', type=<ColInfoTypeEnum.default: 1>, column_label='Tool calls', column_align='right', column_width=None), ColInfo(var='minutes', type=<ColInfoTypeEnum.default: 1>, column_label='Elapsed minutes', column_align='right', column_width=None), ColInfo(var='report_file', type=<ColInfoTypeEnum.default: 1>, column_label='REPORT.md', column_align='left', column_width=None), ColInfo(var='friction_file', type=<ColInfoTypeEnum.default: 1>, column_label='FRICTION.md', column_align='left', column_width=None), ColInfo(var='procedural_messages', type=<ColInfoTypeEnum.default: 1>, column_label='Procedural messages', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10bc22e10>, _spanners=Spanners([]), _heading=Heading(title='What each analyst spent', subtitle='Claude Haiku agents, one per library', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11316a810>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11316a5d0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x113169c70>, _formats=[<great_tables._gt_data.FormatInfo object at 0x11316a540>, <great_tables._gt_data.FormatInfo object at 0x11316a210>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='

## Scorecard

The ten dimensions of the knowledge-discovery scorecard
(`docs/scorecard/scorecard.md`) are scored 1 to 5. Six are measured with the
thresholds fixed in `PROTOCOL.md` before any analyst started; D2, D3, D7 and
D8 are judged from each analyst's report, friction log and pipeline against
the scorecard's anchors, with the evidence in the table that follows the
scorecard. On this battery D1 (relationship discovery) means causal
structure: valid adjustment sets, correct roles, effect modifiers, channels
and named overlap.

In [14]:
def d1(lib):
    s = summary[lib]
    alt = [per[lib][r] for r in ALT if per[lib][r]["ran"]]
    if not alt:
        return 1.0
    valid_all = all(r["roles"]["adjustment_set_valid"] for r in alt)
    if s["role_accuracy"] < 0.6 or s["adjustment_valid_share"] < 0.5:
        return 2.0
    if not (valid_all and s["role_accuracy"] >= 0.6):
        return 2.0
    four = s["modifier_tpr"] >= 0.5 and s["modifier_fdr"] <= 0.25 and (s["null_false_claims"] or 0) <= 1
    if not four:
        return 3.0
    return 5.0 if s["channels_correct"] == 1.0 and s["overlap_named"] == 1.0 else 4.0


def d4(lib):
    c = summary[lib].get("coverage_all_queries")
    if c is None:
        return 1.0
    return 2.0 if c < 0.70 else 3.0 if (c < 0.82 or c > 0.98) else 4.0


def d5(lib):
    t = runlogs[lib].get("followup", {}).get("times") or []
    if not t:
        return 1.0
    m = float(np.median(t))
    return 1.0 if m > 60 else 2.0 if m > 10 else 3.0 if m > 1 else 4.0


def d6(lib):
    s = summary[lib]
    if s["ran"] < s["replicates"]:
        return 1.0
    small = [per[lib][r] for r in ("S1", "S2")]
    full = [per[lib][r] for r in ALT if r.startswith("H")]
    ate_cov_small = np.mean([r["queries"]["ATE"]["covered"] for r in small])
    pehe_small = np.mean([r["cate"]["pehe"] for r in small])
    pehe_full = np.mean([r["cate"]["pehe"] for r in full])
    if ate_cov_small == 0 or pehe_small > 2 * pehe_full:
        return 2.0
    ln = runlogs[lib].get("large_n", {})
    big = ln.get("ok") and (ln.get("seconds") or 1e9) <= 300
    return 4.0 if big else 3.0


def d9(lib):
    s = summary[lib]
    c = s.get("cate_coverage")
    if c is None:
        return 1.0
    if not 0.80 <= c <= 0.97:
        return 2.0
    small_cov = np.mean([per[lib][r]["queries"]["ATE"]["covered"] for r in ("S1", "S2")])
    return 4.0 if small_cov >= 0.8 else 3.0


def tuning_log(lib):
    for path in (RUNS[lib] / "tuning_log.csv", RUNS[lib] / "dev_out" / "tuning_log.csv"):
        if path.exists():
            return path
    return None


def d10(lib):
    g = runlogs[lib].get("repro_grader_vs_grader", {})
    a = runlogs[lib].get("repro_grader_vs_analyst", {})
    same = g.get("findings_equal") is True and g.get("pred_max_abs_diff") == 0.0
    if not same:
        return 1.0
    if tuning_log(lib) is None:
        return 2.0
    diff = a.get("pred_max_abs_diff")
    matches = a.get("findings_equal") is True and isinstance(diff, float) and diff < 1e-9
    return 4.0 if matches else 3.0


measured = {"D1": d1, "D4": d4, "D5": d5, "D6": d6, "D9": d9, "D10": d10}
W_AUTO = {"D1": 2, "D2": 1, "D3": 2, "D4": 1, "D5": 2, "D6": 1, "D7": 0.5, "D8": 2, "D9": 2, "D10": 1}
W_SEMI = {"D1": 2, "D2": 1, "D3": 1, "D4": 2, "D5": 1, "D6": 1, "D7": 2, "D8": 0.5, "D9": 1, "D10": 1}
sheet = []
for d in dims:
    row = {"dimension": f"{d} {names[d]}", "basis": "measured" if d in measured else "judged"}
    for lib in LIBS:
        row[lib] = measured[d](lib) if d in measured else JUDGED[lib][d]["score"]
    sheet.append(row)
for label, w in (("Weighted, fully automated", W_AUTO), ("Weighted, semi-automated", W_SEMI)):
    row = {"dimension": label, "basis": "aggregate"}
    for lib in LIBS:
        row[lib] = sum(w[r["dimension"].split()[0]] * r[lib] for r in sheet[:10]) / sum(w.values())
    sheet.append(row)
scorecard = pl.DataFrame(sheet)
gt = (GT(scorecard, rowname_col="dimension")
      .tab_header(title="Knowledge-discovery scorecard (docs/scorecard/scorecard.md), causal battery",
                  subtitle="1-5 per dimension; measured rows apply PROTOCOL.md's thresholds, judged rows are "
                           "argued in the table below")
      .cols_label(basis="Basis", **LIBS)
      .fmt_number(list(LIBS), decimals=1)
      .tab_source_note("Weights from the scorecard's mode table. The aggregates are shown only with the "
                       "profile above them."))
for i, r in enumerate(sheet[:10]):
    vals = [r[lib] for lib in LIBS]
    winners = [lib for lib in LIBS if r[lib] == max(vals)]
    gt = gt.tab_style(style.text(weight="bold"), loc.body(columns=winners, rows=[i]))
gt

GT(_tbl_data=shape: (12, 7)
┌────────────────────────────┬───────────┬──────────┬──────────┬──────────┬─────────────┬──────────┐
│ dimension                  ┆ basis     ┆ mboostrs ┆ econml   ┆ doubleml ┆ statsmodels ┆ lightgbm │
│ ---                        ┆ ---       ┆ ---      ┆ ---      ┆ ---      ┆ ---         ┆ ---      │
│ str                        ┆ str       ┆ f64      ┆ f64      ┆ f64      ┆ f64         ┆ f64      │
╞════════════════════════════╪═══════════╪══════════╪══════════╪══════════╪═════════════╪══════════╡
│ D1 Relationship discovery  ┆ measured  ┆ 2.0      ┆ 3.0      ┆ 2.0      ┆ 2.0         ┆ 2.0      │
│ D2 Causal humility         ┆ judged    ┆ 5.0      ┆ 5.0      ┆ 5.0      ┆ 5.0         ┆ 5.0      │
│ D3 Explanation quality     ┆ judged    ┆ 4.0      ┆ 4.0      ┆ 3.5      ┆ 4.0         ┆ 3.0      │
│ D4 Importance              ┆ measured  ┆ 4.0      ┆ 3.0      ┆ 4.0      ┆ 3.0         ┆ 3.0      │
│ quantification             ┆           ┆          ┆          ┆          ┆             ┆          │
│ D5 Iteration speed         ┆ measured  ┆ 3.0      ┆ 3.0      ┆ 2.0      ┆ 4.0         ┆ 3.0      │
│ …                          ┆ …         ┆ …        ┆ …        ┆ …        ┆ …           ┆ …        │
│ D8 Agent-friendliness      ┆ judged    ┆ 3.5      ┆ 3.0      ┆ 3.0      ┆ 3.0         ┆ 2.5      │
│ D9 Uncertainty honesty     ┆ measured  ┆ 2.0      ┆ 4.0      ┆ 4.0      ┆ 4.0         ┆ 3.0      │
│ D10 Reproducibility        ┆ measured  ┆ 4.0      ┆ 1.0      ┆ 1.0      ┆ 4.0         ┆ 1.0      │
│ Weighted, fully automated  ┆ aggregate ┆ 3.206897 ┆ 3.344828 ┆ 2.931034 ┆ 3.534483    ┆ 2.827586 │
│ Weighted, semi-automated   ┆ aggregate ┆ 3.26     ┆ 3.24     ┆ 2.96     ┆ 3.32        ┆ 2.74     │
└────────────────────────────┴───────────┴──────────┴──────────┴──────────┴─────────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10bc23a40>, _boxhead=Boxhead([ColInfo(var='dimension', type=<ColInfoTypeEnum.stub: 2>, column_label='dimension', column_align='left', column_width=None), ColInfo(var='basis', type=<ColInfoTypeEnum.default: 1>, column_label='Basis', column_align='left', column_width=None), ColInfo(var='mboostrs', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs 0.1.3', column_align='right', column_width=None), ColInfo(var='econml', type=<ColInfoTypeEnum.default: 1>, column_label='EconML 0.17.0', column_align='right', column_width=None), ColInfo(var='doubleml', type=<ColInfoTypeEnum.default: 1>, column_label='DoubleML 0.11.4', column_align='right', column_width=None), ColInfo(var='statsmodels', type=<ColInfoTypeEnum.default: 1>, column_label='statsmodels 0.15.0', column_align='right', column_width=None), ColInfo(var='lightgbm', type=<ColInfoTypeEnum.default: 1>, column_label='LightGBM 4.7.0', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x113184320>, _spanners=Spanners([]), _heading=Heading(title='Knowledge-discovery scorecard (docs/scorecard/scorecard.md), causal battery', subtitle="1-5 per dimension; measured rows apply PROTOCOL.md's thresholds, judged rows are argued in the table below", preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x113184cb0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x113184c50>, _source_notes=["Weights from the scorecard's mode table. The aggregates are shown only with the profile above them."], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=['econml'], rows=[0], mask=None), grpname=None, colname='econml', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=['mboostrs', 'econml', 'doubleml', 'statsmodels', 'lightgbm'], rows=[1], mask=None), grpname=None, colname='mboostrs', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=N

## Judged dimensions

In [15]:
jrows = []
for d in ["D2", "D3", "D7", "D8"]:
    for lib in LIBS:
        jrows.append({"dimension": f"{d} {names[d]}", "library": LIBS[lib],
                      "score": JUDGED[lib][d]["score"], "evidence": JUDGED[lib][d]["evidence"]})
GT(pl.DataFrame(jrows), rowname_col="library", groupname_col="dimension").tab_header(
    title="Judged dimensions and their evidence",
    subtitle="Scored by the coordinator from REPORT.md, FRICTION.md and run.py against the scorecard anchors") \
    .fmt_number(["score"], decimals=1)

GT(_tbl_data=shape: (20, 4)
┌───────────────────────┬────────────────────┬───────┬─────────────────────────────────┐
│ dimension             ┆ library            ┆ score ┆ evidence                        │
│ ---                   ┆ ---                ┆ ---   ┆ ---                             │
│ str                   ┆ str                ┆ f64   ┆ str                             │
╞═══════════════════════╪════════════════════╪═══════╪═════════════════════════════════╡
│ D2 Causal humility    ┆ mboostrs 0.1.3     ┆ 5.0   ┆ REPORT section 7 gives a drop-… │
│ D2 Causal humility    ┆ EconML 0.17.0      ┆ 5.0   ┆ REPORT section 7 refits with e… │
│ D2 Causal humility    ┆ DoubleML 0.11.4    ┆ 5.0   ┆ REPORT section 7 refits with e… │
│ D2 Causal humility    ┆ statsmodels 0.15.0 ┆ 5.0   ┆ REPORT section 7 gives a parti… │
│ D2 Causal humility    ┆ LightGBM 4.7.0     ┆ 5.0   ┆ REPORT section 7 gives the bia… │
│ …                     ┆ …                  ┆ …     ┆ …                               │
│ D8 Agent-friendliness ┆ mboostrs 0.1.3     ┆ 3.5   ┆ The analyst built the pipeline… │
│ D8 Agent-friendliness ┆ EconML 0.17.0      ┆ 3.0   ┆ Delivered all eight items with… │
│ D8 Agent-friendliness ┆ DoubleML 0.11.4    ┆ 3.0   ┆ Delivered every item without a… │
│ D8 Agent-friendliness ┆ statsmodels 0.15.0 ┆ 3.0   ┆ Delivered all eight items with… │
│ D8 Agent-friendliness ┆ LightGBM 4.7.0     ┆ 2.5   ┆ The analyst stopped twice with… │
└───────────────────────┴────────────────────┴───────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x113186150>, _boxhead=Boxhead([ColInfo(var='dimension', type=<ColInfoTypeEnum.row_group: 3>, column_label='dimension', column_align='left', column_width=None), ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None), ColInfo(var='score', type=<ColInfoTypeEnum.default: 1>, column_label='score', column_align='right', column_width=None), ColInfo(var='evidence', type=<ColInfoTypeEnum.default: 1>, column_label='evidence', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x113185f40>, _spanners=Spanners([]), _heading=Heading(title='Judged dimensions and their evidence', subtitle='Scored by the coordinator from REPORT.md, FRICTION.md and run.py against the scorecard anchors', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x113186810>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x113186660>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x113186450>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1131858e0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=Op

(credit-where-it-is-due)=
## Library results

- **statsmodels** gave the most accurate average and per-applicant effects
  with nothing more than logistic and least-squares fits, cross-fitted by
  hand, an R-learner written as a weighted least-squares fit, and a full
  bootstrap. Its report proposes the design that would test the claim (a
  randomized encouragement arm).
- **EconML** gave the most accurate structure: every confounder in the
  adjustment set on every replicate, the best ranking and no false claims on
  the no-effect data, from `LinearDML` with inverse response weights passed as
  sample weights.
- **DoubleML** handled the missing outcomes directly (`DoubleMLSSM`) and had
  the best all-query coverage among the libraries that ran once (0.88).
- **mboostrs** had the best coverage of the effect queries (0.91) with no
  false claims on the no-effect data, reproduced exactly, and was the only
  pipeline that ranked applicants within the overlap region by construction.

(what-the-runs-showed-about-mboostrs)=
## mboostrs API changes and remaining defects

Each mboostrs analyst's friction log was turned into library changes between
runs: issues #416, #420 and #423, eleven pull requests in all. The changes
the fourth analyst used: `aipw(outcome_missing="ipw")` for the missing
outcomes, `target="overlap"` as a check, the step-cap doubling, `ci_test`
for roles (with the collinear experience column handled), the sensitivity
functions, and `rank_by_effect`. The defects it still met are in #429: `aipw`
does not read the defaults stored in `CausalSpec`, its relayed warnings advise
arguments it does not take, it accepts no nuisance formula (the enrolment
propensity has a step at the eligibility limit), the cross-validation results
have no risk path, and there is no instrumental-variable estimator.

Across the four mboostrs runs the weighted score moved from 3.03 to 3.21
(fully automated) and from 3.06 to 3.26 (semi-automated); the record of all
runs is `benchmarks/causal_discovery_before.ipynb`.

The change of the `aipw` default to `smooth=True` (PR #712) alters none of the reruns: each reported `aipw` call passes `smooth=` explicitly. The reported fit in `rerun3/` used `smooth=False`, because its `choose_smooth` step picked linear terms on that data (`smooth_baseline` is false in `rerun3/mboostrs/dev_out/run_details.json`), and its ATE of 4,930 dollars comes from that fit. Every reported fit in `rerun/`, `rerun2/`, `rerun4/`, `rerun5/` and `rerun6/` used `smooth=True`. Side fits with `smooth=False` also appear in the tuning logs of `rerun3/` (C05) and `rerun4/` (T02).

## Departures from the protocol

- The LightGBM analyst stopped twice with deliverables missing, because it
  ended its turn while its own background job ran. It received two procedural
  messages ("continue and finish the assignment as written"; "run long
  commands in the foreground"), with no help on the analysis.
- The EconML analyst reported that a session note showed its working
  directory as the statsmodels run directory, because the grader's shell was
  in that directory. It stated it did not open it.
- The agent harness refused some report files (marked in the effort table);
  their text was saved unchanged from the analysts' final messages.
- The protocol named one mboostrs rerun. Each rerun's friction log found new
  library defects, so there were three fix rounds and three reruns. All four
  mboostrs runs are reported.
- The fourth analyst's evaluation started from its delivered `run.py` before
  its final message arrived; the file's hash was checked unchanged afterwards.
- The agent guide's causal topic was written after the first run, from that
  run's friction log. It is general guidance for observational studies, but
  its example of post-treatment selection (survey response) is also one of
  this benchmark's traps.

## Limits of this comparison

- One analyst per library and run. An agent's choices move the results as
  much as the library does: the four mboostrs analysts, with the same brief,
  wrote four different estimators.
- Ten hidden replicates; a coverage of 0.875 is 7 of 8.
- The pipelines ran on a shared machine, so run times are upper bounds.
- The generating model has no unmeasured confounder, so the sensitivity
  analyses are graded on how they are reported, not on whether they detect
  one.
- The dollar-scale modifiers follow from a proportional effect on earnings;
  an analysis on the log scale would find a different list (education and age
  instead of prior earnings), which the brief's dollar framing rules out.